# 03 — Deep Learning Models (Stage 3)

Train and compare Stage 3 deep learning (recurrent sequence) models using the shared PyTorch pipeline.

Models compared: Bidirectional LSTM (BiLSTM), Bidirectional GRU (GRU).

In [ ]:
import subprocess
import sys
from pathlib import Path

INSTALL_ON_REMOTE = False  # set True on Colab/Kaggle for first run

candidate = Path.cwd().resolve()
for path in (candidate, *candidate.parents):
    if (path / "pyproject.toml").exists():
        PROJECT_ROOT = path
        break
else:
    raise FileNotFoundError(
        "Could not find project root. Open this notebook from the repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if INSTALL_ON_REMOTE:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", ".[dev,notebook,deep]"],
        cwd=PROJECT_ROOT,
    )

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

from src.models.deep_compare import compare_deep_models
from utils.config import load_config

sns.set_theme(style="whitegrid")

DEEP_LEARNING_CONFIG = PROJECT_ROOT / "configs/deep_learning.yaml"
print("CUDA Available:", torch.cuda.is_available())

## Stage 3 — Train and Compare Recurrent Models

We will train the Bidirectional LSTM and Bidirectional GRU models on the WELFake dataset. The training pipeline uses an early stopping mechanism based on validation performance to prevent overfitting.

In [ ]:
comparison = compare_deep_models(DEEP_LEARNING_CONFIG)
comparison

## Visualize Model Performance

Compare the F1 scores and training times of the BiLSTM and GRU models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot F1 Score
sns.barplot(data=comparison, x="model", y="f1", ax=axes[0], palette="viridis")
axes[0].set_title("Model comparison (F1 score)")
axes[0].set_xlabel("Model")
axes[0].set_ylabel("F1 Score")
axes[0].set_ylim(0.8, 1.0)

# Plot Training Time
sns.barplot(data=comparison, x="model", y="training_seconds", ax=axes[1], palette="magma")
axes[1].set_title("Model comparison (Training Time)")
axes[1].set_xlabel("Model")
axes[1].set_ylabel("Training Time (seconds)")

plt.tight_layout()
plt.show()

## Analyze Training Curves

We can load the detailed training logs from `model_comparison.json` to plot the training and validation loss curves over the epochs.

In [ ]:
config = load_config(DEEP_LEARNING_CONFIG)
comparison_json_path = Path(config["output"]["model_dir"]) / config["output"]["comparison_json"]
if not comparison_json_path.is_absolute():
    comparison_json_path = PROJECT_ROOT / comparison_json_path

with open(comparison_json_path, "r") as f:
    detailed_results = json.load(f)

plt.figure(figsize=(10, 6))
for result in detailed_results:
    model_name = result["model_name"]
    history = pd.DataFrame(result["history"])
    plt.plot(history["epoch"], history["train_loss"], "o--", label=f"{model_name} (Train Loss)")
    plt.plot(history["epoch"], history["f1"], "s-", label=f"{model_name} (Val F1)")

plt.title("Training Loss & Validation F1 Curves")
plt.xlabel("Epoch")
plt.ylabel("Score / Loss")
plt.legend()
plt.grid(True)
plt.show()